# 1D Preprocessing

In [ ]:
import os
import sys
# Add the parent directory to the Python path to allow module imports
sys.path.append(os.path.dirname(os.getcwd()))

import json
import numpy as np
import pandas as pd
import shutil
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm

from modules.datasets import ICBHIAudioDataset, KAUHAudioDataset
from modules.lungsound import LungSoundAudio
from modules.transforms import *

In [ ]:
DATA_PATH = Path(os.path.join(os.path.dirname(os.getcwd()), "data"))
RAW_DATA_FOLDER = DATA_PATH / "raw"
INTERIM_DATA_FOLDER = DATA_PATH / "interim"

if not os.path.exists(RAW_DATA_FOLDER):
    raise FileNotFoundError(f"Raw data folder not found at {RAW_DATA_FOLDER}. Please ensure the original data was already downloaded and placed in the correct location.")

if not os.path.exists(INTERIM_DATA_FOLDER):
    os.makedirs(INTERIM_DATA_FOLDER)
    print(f"Created interim data folder at {INTERIM_DATA_FOLDER}.")
else:
    if len(os.listdir(INTERIM_DATA_FOLDER)) > 0:
        print(f"[WARNING] Interim data folder already exist and is not empty ({INTERIM_DATA_FOLDER}). Consider deleting it to run the preprocessing step again.")

## Interim

In [ ]:
TARGET_SR = 22050       # Hz
WINDOW_LENGTH = 5       # seconds
STANDARD_HOP_LENGTH = 2.5 # overlap of 3 seconds between windows
COPD_HOP_LENGTH = 5     # no overlap
# NOTE: For COPD, we will not apply any overlap between windows to undersample this class, which is overrepresented in the dataset.

def preprocess_audios(original_data_path: Path, preprocessed_data_path: Path):
    """
    Preprocesses the original data and saves the preprocessed data to the specified location using multiple feature extractors.
    Args:
        original_data_path (Path): Path to the original raw data.
        preprocessed_data_path (Path): Path where the preprocessed data will be saved.
    """
    datasets = [
        ICBHIAudioDataset(original_data_path),
        KAUHAudioDataset(original_data_path)
    ]

    # Iterate through each dataset and apply preprocessing to the audio files
    for dataset in datasets:
        df = dataset.data
        new_rows = []
        for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Preprocessing {dataset.__class__.__name__}"):
            file_path = dataset.root / row["FilePath"]
            # Load the audio file using the LungSound class
            audio = LungSoundAudio(str(file_path))
            # Apply audio transforms
            # 1. Split the audio into windows of fixed duration
            diagnosis = row["Diagnosis"]
            HOP_LENGTH = COPD_HOP_LENGTH if diagnosis == "COPD" else STANDARD_HOP_LENGTH
            cropped_audios = Window(window_length=WINDOW_LENGTH, hop_length=HOP_LENGTH)(audio)
            for i, cropped_audio in enumerate(cropped_audios):
                # 2. Resample the audio to the target sampling rate
                resampled_audio = Resample(target_sr=TARGET_SR)(cropped_audio)
                # 3. Normalize the audio
                normalized_audio = NormalizeAudio()(resampled_audio)
                # Save the preprocessed audio to the new location
                start = i * HOP_LENGTH
                end = start + WINDOW_LENGTH
                preprocessed_file_name = f"{file_path.stem}_clip-{start:.1f}s-{end:.1f}s.wav"
                preprocessed_file_path = preprocessed_data_path / dataset.name / diagnosis / preprocessed_file_name 
                preprocessed_file_path.parent.mkdir(parents=True, exist_ok=True)
                sf.write(preprocessed_file_path, normalized_audio.audio, normalized_audio.sr)

                # Create a new row for the preprocessed audio file with the same metadata as the original row
                new_row = row.copy()
                relpath = preprocessed_file_path.relative_to(preprocessed_data_path)
                new_row["FilePath"] = relpath
                new_rows.append(new_row)
            
        # Save the new rows to a new dataframe and then to a new CSV file
        data_path = preprocessed_data_path / dataset.name / f"metadata.csv"
        new_df = pd.DataFrame(new_rows)
        new_df.to_csv(data_path, index=False)

        print(f"Processed {len(df)} original audio files from {dataset.__class__.__name__}.")
        print(f"Created {len(new_rows)} preprocessed audio files for {dataset.__class__.__name__}.")

        # Save the audio preprocessing informations to a JSON file
        audio_transforms = {
            Window.__name__: {"params": vars(Window(window_length=WINDOW_LENGTH, hop_length=STANDARD_HOP_LENGTH))},
            Resample.__name__: {"params": vars(Resample(target_sr=TARGET_SR))},
            NormalizeAudio.__name__: {"params": vars(NormalizeAudio())},
        }
        json_path = preprocessed_data_path / dataset.name / "audio_preprocessing.json"
        notes = f"For COPD, we applied a different hop length of {COPD_HOP_LENGTH} seconds to undersample this class."
        with open(json_path, "w") as f:
            json.dump({"audio_transforms": audio_transforms, "NOTES": notes}, f, indent=4)

    print("-------------------------------------------------------")
    print(f"Preprocessing for {original_data_path.name} completed.")
    print(f"Data saved to {os.path.relpath(preprocessed_data_path, start=os.getcwd())}")

In [ ]:
preprocess_audios(RAW_DATA_FOLDER, INTERIM_DATA_FOLDER)